In [4]:
import pandas as pd

def merge_lncRNA_features_with_names(
    feature_files,
    seq_file="二级结构.csv",
    output_file="lncRNA_all_features.csv"
):
    # 1. 读取 lncRNA 名称
    seq_df = pd.read_csv(seq_file)  # 里面有列 "lncRNA"
    merged_df = pd.DataFrame()
    merged_df["lncRNA"] = seq_df["lncRNA"]

    # 2. 按顺序拼接各个特征文件
    for i, file in enumerate(feature_files, start=1):
        df = pd.read_csv(file, header=None)  # 无列名
        df.columns = [f"Feature_{i}_{j}" for j in range(df.shape[1])]  # 自动命名
        merged_df = pd.concat([merged_df, df], axis=1)

    # 3. 保存结果
    merged_df.to_csv(output_file, index=False)
    print(f"✅ 拼接完成，保存到 {output_file}, shape={merged_df.shape}")
    return merged_df


# 用法
files = [
    "Word2Vec.csv",
    "GC_related.csv",
    "CTD_sequence.csv",
    "Protpar.csv",
    "Feature_i.csv"
]
merge_lncRNA_features_with_names(files)



✅ 拼接完成，保存到 lncRNA_all_features.csv, shape=(2179, 6)


,lncRNA,Feature_1_0,Feature_2_0,Feature_3_0,Feature_4_0,Feature_5_0
0,RP11-654A16.3,-0.001096544787287712 -2.385140396654606e-05 0...,0.605442176870031 0.5663265306108913 0.6428571...,0.195578231292517 0.19727891156462585 0.307823...,0.5564162325198793 0.7928678988563999 0.330109...,0.012329931972789115 0.012329931972789115 0.01...
1,LINC00963,0.024843935644947017 -0.15254130511967395 0.20...,0.5354017501987736 0.5083532219569609 0.568019...,0.26014319809069214 0.20365950676213207 0.2792...,0.53177131889224 0.5676070678328589 0.28127659...,0.016308671439936355 0.012728719172633254 0.01...
2,LINC00588,-0.030066095666395894 -0.2183456902841173 0.10...,0.4134241245137029 0.4128373450039013 0.417213...,0.29401750972762647 0.2923151750972763 0.22470...,0.4732694539704242 0.9368154417961383 0.216117...,0.018391293774319067 0.018269698443579768 0.01...
3,LINC00599,0.023629356024077456 -0.1922883618216755 0.128...,0.568085106382689 0.5382165605090673 0.6325878...,0.20212765957446807 0.2297872340425532 0.27021...,0.3588565944612011 0.33750327488666304 0.43380...,0.012632978723404254 0.014361702127659574 0.01...
4,RP11-217B7.2,-0.03052400276904753 -0.20061243218126265 0.13...,0.5339999999999093 0.551999999999584 0.5399999...,0.25933333333333336 0.206 0.24666666666666667 ...,0.4428062719177984 0.6442214945702964 0.652855...,0.01625 0.012875 0.017958333333333333 0.015416...
...,...,...,...,...,...,...
2174,RP11-521O16.2,-0.03866274181092059 -0.20649398404867092 0.12...,0.41756032171589136 0.4229222520109305 0.42493...,0.2848525469168901 0.29736371760500446 0.20688...,0.4445267861774579 0.5844077708707377 0.382403...,0.017817247542448615 0.01858523235031278 0.013...
2175,RP11-322F10.2,-0.011456287498358982 -0.10338475836191541 0.1...,0.31016949152671075 0.24873096447210696 0.3502...,0.35084745762711866 0.3389830508474576 0.15254...,0.3027646537723313 0.6878002681453278 0.599018...,0.021927966101694917 0.0211864406779661 0.0098...
2176,LINC00545,-0.003837248959245494 -0.17629312515360504 0.1...,0.46701388888911793 0.4843750000003255 0.5 0.4...,0.2517361111111111 0.2795138888888889 0.215277...,0.526349006741488 0.8264506446545749 0.3307654...,0.015842013888888888 0.017469618055555556 0.01...
2177,RP11-672A2.6,0.0012772268554970918 -0.20876600749089744 0.1...,0.5054151624548476 0.48375451263561364 0.52707...,0.2611311672683514 0.2322503008423586 0.217809...,0.6227705835431363 0.8080233821225418 0.427466...,0.016320697954271962 0.014590854392298435 0.01...


In [1]:
# @Date:   2024/5/28 18:49
# description: RNA一维、二维特征提取
# Feature processing

import numpy as np
import re
import math
from sklearn import ensemble
from gensim.models import Word2Vec
from collections import defaultdict
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from _07_GCcounts import GCconder
from CTDcode import CTDcoder
from _06_proparcoder import ProtPar
from sklearn.feature_extraction.text import TfidfVectorizer


# np.random.seed(1337) # random seed
#
separator, Sequencekmertotal, SequenceGgaptotal, Structurekmertotal, StructureGgaptotal = ' ', 3, 3, 3, 3

def SequencekmerExtract(sequence, totalkmer):
    #可以不用替换了
    sequence = sequence.replace('U', 'T')
    character = 'ATCG'
    sequencekmer = ''
    for k in range(totalkmer):
        kk = k + 1
        sk = len(sequence) - kk + 1
        wk = 1 / (4 ** (totalkmer - kk))
        # 1-mer
        if kk == 1:
            for char11 in character:
                s1 = char11
                f1 = wk * sequence.count(s1) / sk
                string1 = str(f1) + separator
                sequencekmer = sequencekmer + string1
        # 2-mer
        if kk == 2:
            for char21 in character:
                for char22 in character:
                    s2 = char21 + char22
                    numkmer2 = 0
                    for lkmer2 in range(len(sequence) - kk + 1):
                        if sequence[lkmer2] == s2[0] and sequence[lkmer2 + 1] == s2[1]:
                            numkmer2 = numkmer2 + 1
                    f2 = wk * numkmer2 / sk
                    string2 = str(f2) + separator
                    sequencekmer = sequencekmer + string2
        # 3-mer
        if kk == 3:
            for char31 in character:
                for char32 in character:
                    for char33 in character:
                        s3 = char31 + char32 + char33
                        numkmer3 = 0
                        for lkmer3 in range(len(sequence) - kk + 1):
                            if sequence[lkmer3] == s3[0] and sequence[lkmer3 + 1] == s3[1] and sequence[lkmer3 + 2] == s3[2]:
                                numkmer3 = numkmer3 + 1
                        f3 = wk * numkmer3 / sk
                        string3 = str(f3) + separator
                        sequencekmer = sequencekmer + string3
        # 4-mer
        if kk == 4:
            for char41 in character:
                for char42 in character:
                    for char43 in character:
                        for char44 in character:
                            s4 = char41 + char42 + char43 + char44
                            numkmer4 = 0
                            for lkmer4 in range(len(sequence) - kk + 1):
                                if sequence[lkmer4] == s4[0] and sequence[lkmer4 + 1] == s4[1] and sequence[lkmer4 + 2] == s4[2] and sequence[lkmer4 + 3] == s4[3]:
                                    numkmer4 = numkmer4 + 1
                            f4 = wk * numkmer4 / sk
                            string4 = str(f4) + separator
                            sequencekmer = sequencekmer + string4
        # 5-mer
        if kk == 5:
            for char51 in character:
                for char52 in character:
                    for char53 in character:
                        for char54 in character:
                            for char55 in character:
                                s5 = char51 + char52 + char53 + char54 + char55
                                numkmer5 = 0
                                for lkmer5 in range(len(sequence) - kk + 1):
                                    if sequence[lkmer5] == s5[0] and sequence[lkmer5 + 1] == s5[1] and sequence[lkmer5 + 2] == s5[2] and sequence[lkmer5 + 3] == s5[3] and sequence[lkmer5 + 4] == s5[4]:
                                        numkmer5 = numkmer5 + 1
                                f5 = wk * numkmer5 / sk
                                string5 = str(f5) + separator
                                sequencekmer = sequencekmer + string5
        # 6-mer
        if kk == 6:
            for char61 in character:
                for char62 in character:
                    for char63 in character:
                        for char64 in character:
                            for char65 in character:
                                for char66 in character:
                                    s6 = char61 + char62 + char63 + char64 + char65 + char66
                                    numkmer6 = 0
                                    for lkmer6 in range(len(sequence) - kk + 1):
                                        if sequence[lkmer6] == s6[0] and sequence[lkmer6 + 1] == s6[1] and sequence[lkmer6 + 2] == s6[2] and sequence[lkmer6 + 3] == s6[3] and sequence[lkmer6 + 4] == s6[4] and sequence[lkmer6 + 5] == s6[5]:
                                            numkmer6 = numkmer6 + 1
                                    f6 = wk * numkmer6 / sk
                                    string6 = str(f6) + separator
                                    sequencekmer = sequencekmer + string6
    return sequencekmer

def SequenceGgapExtract(sequence, totalGgap):
    sequence = sequence.replace('U', 'T')
    character = 'ATCG'
    sequenceGgap = ''
    for k in range(totalGgap):
        kk = k + 1
        sk = len(sequence) - kk + 1
        wk = 1 / (4 ** (totalGgap - kk))
        if kk == 1:
            for char11 in character:
                for char12 in character:
                    num1 = 0
                    for l1 in range(len(sequence) - kk - 1):
                        if sequence[l1] == char11 and sequence[l1 + kk + 1] == char12:
                            num1 = num1 + 1
                    f1 = wk * num1 / sk
                    string1 = str(f1) + separator
                    sequenceGgap = sequenceGgap + string1
        if kk == 2:
            for char21 in character:
                for char22 in character:
                    num2 = 0
                    for l2 in range(len(sequence) - kk - 3):
                        if sequence[l2] == char21 and sequence[l2 + kk + 1] == char22:
                            num2 = num2 + 1
                    f2 = wk * num2 / sk
                    string2 = str(f2) + separator
                    sequenceGgap = sequenceGgap + string2
        if kk == 3:
            for char31 in character:
                for char32 in character:
                    num3 = 0
                    for l3 in range(len(sequence) - kk - 3):
                        if sequence[l3] == char31 and sequence[l3 + kk + 1] == char32:
                            num3 = num3 + 1
                    f3 = wk * num3 / sk
                    string3 = str(f3) + separator
                    sequenceGgap = sequenceGgap + string3
        if kk == 4:
            for char41 in character:
                for char42 in character:
                    num4 = 0
                    for l4 in range(len(sequence) - kk - 3):
                        if sequence[l4] == char41 and sequence[l4 + kk + 1] == char42:
                            num4 = num4 + 1
                    f4 = wk * num4 / sk
                    string4 = str(f4) + separator
                    sequenceGgap = sequenceGgap + string4
        if kk == 5:
            for char51 in character:
                for char52 in character:
                    num5 = 0
                    for l5 in range(len(sequence) - kk - 3):
                        if sequence[l5] == char51 and sequence[l5 + kk + 1] == char52:
                            num5 = num5 + 1
                    f5 = wk * num5 / sk
                    string5 = str(f5) + separator
                    sequenceGgap = sequenceGgap + string5
    return sequenceGgap

def StructurekmerExtract(structure, totalkmer):
    character = ').'
    structurekmer = '' # 特征

    sp = structure.split()
    ssf = sp[0]
    ssf = ssf.replace('(', ')')
    for k in range(totalkmer):
        kk = k + 1
        sk = len(ssf) - kk + 1
        wk = 1 / (2 ** (totalkmer - kk))
        # 1-mer
        if kk == 1:
            for char11 in character:
                s1 = char11
                f1 = wk * ssf.count(s1) / sk
                string1 = str(f1) + separator
                structurekmer = structurekmer + string1
        # 2-mer
        if kk == 2:
            for char21 in character:
                for char22 in character:
                    s2 = char21 + char22
                    numkmer2 = 0
                    for lkmer2 in range(len(ssf) - kk + 1):
                        if ssf[lkmer2] == s2[0] and ssf[lkmer2 + 1] == s2[1]:
                            numkmer2 = numkmer2 + 1
                    f2 = wk * numkmer2 / sk
                    string2 = str(f2) + separator
                    structurekmer = structurekmer + string2
        # 3-mer
        if kk == 3:
            for char31 in character:
                for char32 in character:
                    for char33 in character:
                        s3 = char31 + char32 + char33
                        numkmer3 = 0
                        for lkmer3 in range(len(ssf) - kk + 1):
                            if ssf[lkmer3] == s3[0] and ssf[lkmer3 + 1] == s3[1] and ssf[lkmer3 + 2] == s3[2]:
                                numkmer3 = numkmer3 + 1
                        f3 = wk * numkmer3 / sk
                        string3 = str(f3) + separator
                        structurekmer = structurekmer + string3
        # 4-mer
        if kk == 4:
            for char41 in character:
                for char42 in character:
                    for char43 in character:
                        for char44 in character:
                            s4 = char41 + char42 + char43 + char44
                            numkmer4 = 0
                            for lkmer4 in range(len(ssf) - kk + 1):
                                if ssf[lkmer4] == s4[0] and ssf[lkmer4 + 1] == s4[1] and ssf[lkmer4 + 2] == s4[2] and ssf[lkmer4 + 3] == s4[3]:
                                    numkmer4 = numkmer4 + 1
                            f4 = wk * numkmer4 / sk
                            string4 = str(f4) + separator
                            structurekmer = structurekmer + string4
        # 5-mer
        if kk == 5:
            for char51 in character:
                for char52 in character:
                    for char53 in character:
                        for char54 in character:
                            for char55 in character:
                                s5 = char51 + char52 + char53 + char54 + char55
                                numkmer5 = 0
                                for lkmer5 in range(len(ssf) - kk + 1):
                                    if ssf[lkmer5] == s5[0] and ssf[lkmer5 + 1] == s5[1] and ssf[lkmer5 + 2] == s5[2] and ssf[lkmer5 + 3] == s5[3] and ssf[lkmer5 + 4] == s5[4]:
                                        numkmer5 = numkmer5 + 1
                                f5 = wk * numkmer5 / sk
                                string5 = str(f5) + separator
                                structurekmer = structurekmer + string5
    return structurekmer

def StructureGgapExtract(structure, totalGgap):
    character = ').'
    structureGgap = ''
    sp = structure.split()
    ssf = sp[0]
    ssf = ssf.replace('(', ')')
    for k in range(totalGgap):
        kk = k + 1
        sk = len(ssf) - kk + 1
        wk = 1 / (2 ** (totalGgap - kk))
        if kk == 1:
            for char11 in character:
                for char12 in character:
                    for char13 in character:
                        for char14 in character:
                            num1 = 0
                            for l1 in range(len(ssf) - kk - 3):
                                if ssf[l1] == char11 and ssf[l1 + 1] == char12 and ssf[l1 + kk + 2] == char13 and ssf[l1 + kk + 3] == char14:
                                    num1 = num1 + 1
                            f1 = wk * num1 / sk
                            string1 = str(f1) + separator
                            structureGgap = structureGgap + string1
        if kk == 2:
            for char21 in character:
                for char22 in character:
                    for char23 in character:
                        for char24 in character:
                            num2 = 0
                            for l2 in range(len(ssf) - kk - 3):
                                if ssf[l2] == char21 and ssf[l2 + 1] == char22 and ssf[l2 + kk + 2] == char23 and ssf[l2 + kk + 3] == char24:
                                    num2 = num2 + 1
                            f2 = wk * num2 / sk
                            string2 = str(f2) + separator
                            structureGgap = structureGgap + string2
        if kk == 3:
            for char31 in character:
                for char32 in character:
                    for char33 in character:
                        for char34 in character:
                            num3 = 0
                            for l3 in range(len(ssf) - kk - 3):
                                if ssf[l3] == char31 and ssf[l3 + 1] == char32 and ssf[l3 + kk + 2] == char33 and ssf[l3 + kk + 3] == char34:
                                    num3 = num3 + 1   
                            f3 = wk * num3 / sk
                            string3 = str(f3) + separator
                            structureGgap = structureGgap + string3
        if kk == 4:
            for char41 in character:
                for char42 in character:
                    for char43 in character:
                        for char44 in character:
                            num4 = 0
                            for l4 in range(len(ssf) - kk - 3):
                                if ssf[l4] == char41 and ssf[l4 + 1] == char42 and ssf[l4 + kk + 2] == char43 and ssf[l4 + kk + 3] == char44:
                                    num4 = num4 + 1
                            f4 = wk * num4 / sk
                            string4 = str(f4) + separator
                            structureGgap = structureGgap + string4
        if kk == 5:
            for char51 in character:
                for char52 in character:
                    for char53 in character:
                        for char54 in character:
                            num5 = 0
                            for l5 in range(len(ssf) - kk - 3):
                                if ssf[l5] == char51 and ssf[l5 + 1] == char52 and ssf[l5 + kk + 2] == char53 and ssf[l5 + kk + 3] == char54:
                                    num5 = num5 + 1
                            f5 = wk * num5 / sk
                            string5 = str(f5) + separator
                            structureGgap = structureGgap + string5
    return structureGgap

def ArithmeticLevel(feature1, feature2):
    fpair = ''
    if feature1 != '' and feature2 != '':
        f1, f2 = feature1.strip().split(' '), feature2.strip().split(' ')
        for i in range(len(f1)):
            a = float(f1[i])
            b = float(f2[i])
            c = 50 * (a + b) / 2
            fpair += str(c) + separator
    return fpair

def get_kmers(sequence, k):
    kmers = [sequence[i:i + k] for i in range(len(sequence) - k + 1)]
    return kmers

def get_sequence_vector(sequence, model, k):
    kmers = get_kmers(sequence, k)
    vector_size = model.vector_size
    # 初始化一个零向量
    sequence_vector = np.zeros(vector_size)
    count = 0
    for kmer in kmers:
        if kmer in model.wv:
            sequence_vector += model.wv[kmer]
            count += 1
    if count > 0:
        sequence_vector /= count
    return sequence_vector

def kmer_word2vec(seq, k):
    seq = seq[:-(len(seq) % 3)] if len(seq) % 3 != 0 else seq
    kmers = get_kmers(seq, k)
    sentences = [kmers]
    model = Word2Vec(vector_size=128, window=5, min_count=1, sg=1, negative=5)
    model.build_vocab(sentences)
    model.train(sentences, total_examples=model.corpus_count, epochs=50)
    # 获取RNA序列的特征表示
    sequence_vector = get_sequence_vector(seq, model, k)
    # 转换numpy数组为一个字符串，并用空格分隔
    sequence_vector_str = ' '.join(map(str, sequence_vector))
    return sequence_vector_str


# def get_sequence_vector(sequence, model, k, tfidf_weights):
#     kmers = get_kmers(sequence, k)
#     vector_size = model.vector_size
#     sequence_vector = np.zeros(vector_size)
#     count = 0
#     for kmer in kmers:
#         if kmer in model.wv:
#             # 取对应k-mer的TF-IDF权重
#             tfidf_weight = tfidf_weights.get(kmer, 1.0)
#             sequence_vector += model.wv[kmer] * tfidf_weight
#             count += 1
#     if count > 0:
#         sequence_vector /= count
#     return sequence_vector
#
#
# def kmer_word2vec(seq, k):
#     seq = seq[:-(len(seq) % 3)] if len(seq) % 3 != 0 else seq
#     kmers = get_kmers(seq, k)
#     sentences = [" ".join(kmers)]
#
#     # 计算 TF-IDF 权重
#     tfidf_vectorizer = TfidfVectorizer()
#     tfidf_vectorizer.fit(sentences)
#     tfidf_features = tfidf_vectorizer.transform(sentences)
#
#     # 创建一个词到TF-IDF权重的映射
#     feature_names = tfidf_vectorizer.get_feature_names_out()
#     tfidf_weights = {feature_names[i]: tfidf_features[0, i] for i in range(len(feature_names))}
#
#     # 输出TF-IDF权重进行调试
#     # print("TF-IDF weights:", tfidf_weights)
#
#     model = Word2Vec(vector_size=32, window=5, min_count=1, sg=1, negative=5)
#     model.build_vocab([kmers])
#     model.train([kmers], total_examples=model.corpus_count, epochs=50)
#
#     # 获取RNA序列的特征表示，加入TF-IDF权重
#     sequence_vector = get_sequence_vector(seq, model, k, tfidf_weights)
#
#     # 转换numpy数组为一个字符串，并用空格分隔
#     sequence_vector_str = ' '.join(map(str, sequence_vector))
#     return sequence_vector_str


# def FeatureConstruction(ListPair):
#     Seq_kmer=[]
#     Seq_Ggap=[]
#     Str_kmer =[]
#     Str_Ggap = []
#     Feature=[]
#     DACC=[]
#     CTD=[]
#     for LinePair in ListPair:
#         RNAiname, RNAisequence, RNAistructure = LinePair.strip().split(',')
#         # RNAi sequence k-mer
#         SequenceRNAikmer = SequencekmerExtract(RNAisequence, Sequencekmertotal)
#         # RNAi sequence g-gap
#         SequenceRNAiGgap = SequenceGgapExtract(RNAisequence, SequenceGgaptotal)
#         # RNAi structure k-mer
#         StructureRNAikmer = StructurekmerExtract(RNAistructure, Structurekmertotal)
#         # RNAi structure g-gap
#         StructureRNAiGgap = StructureGgapExtract(RNAistructure, StructureGgaptotal)
#         # RNAi k-mer&word2vec features
#         k_word2vec = kmer_word2vec(RNAisequence, 3)
#         # GC特征：7维    初始化GCconder类，提供序列信息
#         gc_calculator = GCconder(sequence=RNAisequence)
#         gc = gc_calculator.get_gc()
#         gc = ' '.join(map(str, gc))
#         # 转录序列描述CTD：30维
#         ctd_calculator = CTDcoder(sequence=RNAisequence)
#         ctd = ctd_calculator.CTD()
#         ctd = ' '.join(map(str, ctd))
#         # 伪蛋白特征： 5维 (需要后续进行归一化处理，保持量纲统一)
#         protpar = ProtPar(RNAisequence)
#         pro_fea = protpar.get_features()
#         # print(k_word2vec)
#         feature=SequenceRNAikmer + SequenceRNAiGgap + StructureRNAikmer + StructureRNAiGgap
#         DACC.append(dacc_features)
#         Feature.append(f"{feature}")
#         Seq_kmer.append(f"{SequenceRNAikmer}")
#         Seq_Ggap.append(f"{SequenceRNAiGgap}")
#         Str_kmer.append(f"{StructureRNAikmer}")
#         Str_Ggap.append(f"{StructureRNAiGgap}")
#
#     return Seq_kmer, Seq_Ggap, Str_kmer, Str_Ggap, Feature, DACC


pathi='二级结构.csv'   #lncRNA存放格式  名称  人类的序列  二级结构
# list_tv_set = open(pathi, 'r').readlines()
list_tv_set = open(pathi, 'r').readlines()[1:]  # 从第二行开始，跳过表头

CTD = []
Protpar = []
GC = []
Dacc = []
w2v = []
Seq=[]
Strc = []
Feature = []
for line in list_tv_set:
    RNAiname,  RNAisequence, RNAistructure = line.strip().split(',')
    # # RNAi sequence k-mer
    SequenceRNAikmer = SequencekmerExtract(RNAisequence, Sequencekmertotal)
    # # RNAi sequence g-gap
    SequenceRNAiGgap = SequenceGgapExtract(RNAisequence, SequenceGgaptotal)
    # # RNAi structure k-mer
    StructureRNAikmer = StructurekmerExtract(RNAistructure, Structurekmertotal)
    # # RNAi structure g-gap
    StructureRNAiGgap = StructureGgapExtract(RNAistructure, StructureGgaptotal)
    featurei = SequenceRNAikmer + SequenceRNAiGgap + StructureRNAikmer + StructureRNAiGgap
    Feature.append(f"{featurei}")
    # 3_mer + Word2Vec：32维
    k_word2vec = kmer_word2vec(RNAisequence, 3)
    w2v.append(f"{k_word2vec}")
#     # GC特征：7维    初始化GCconder类，提供序列信息
    gc_calculator = GCconder(sequence=RNAisequence)
    gc = gc_calculator.get_gc()
    gc = ' '.join(map(str, gc))
    GC.append(gc)
#     # 转录序列描述CTD：30维
    ctd_calculator = CTDcoder(sequence=RNAisequence)
    ctd = ctd_calculator.CTD()
    ctd = ' '.join(map(str, ctd))
    CTD.append(f"{ctd}")
#     # 伪蛋白特征： 5维 (需要后续进行归一化处理，保持量纲统一)
    protpar = ProtPar(RNAisequence)
    pro_fea = protpar.get_features()
    Protpar.append(pro_fea)

    print(RNAiname)
#
scaler = MinMaxScaler()
Protpar= scaler.fit_transform(Protpar)
pro_str = [' '.join(map(str, features)) for features in Protpar]


# with open ('../data/lnc_old/Seq.csv', 'w') as file:
#     for fea in Seq:
#         file.write(fea+'\n')
# with open ('../data/lnc_old/Str.csv', 'w') as file:
#     for fea in Strc:
#         file.write(fea+'\n')

with open ('Word2Vec.csv','a') as file:
    for fea in w2v:
        file.write(fea+'\n')

with open ('GC_related.csv','a') as file1:
    for gc in GC:
        file1.write(gc+'\n')

with open ('CTD_sequence.csv','a') as file2:
    for gc in CTD:
        file2.write(gc+'\n')
with open ('Protpar.csv','a') as file3:
    for pro in pro_str:
        file3.write(pro+'\n')
with open ('Feature_i.csv','a') as file4:
    for pro in Feature:
        file4.write(pro+'\n')




# pathj='data/miRNA_data.csv'
# tv_set = open(pathj, 'r').readlines()
# CTD_j = []
# Protpar_j = []
# GC_j = []
# Dacc_j = []
# w2v_j = []
# Seq_j=[]
# Str_j = []
# for line in tv_set:
#     RNAiname, RNAisequence, RNAistructure = line.strip().split(',')
#     # RNAi sequence k-mer
#     SequenceRNAikmer = SequencekmerExtract(RNAisequence, Sequencekmertotal)
#     # RNAi sequence g-gap
#     SequenceRNAiGgap = SequenceGgapExtract(RNAisequence, SequenceGgaptotal)
#     # RNAi structure k-mer
#     StructureRNAikmer = StructurekmerExtract(RNAistructure, Structurekmertotal)
#     # RNAi structure g-gap
#     StructureRNAiGgap = StructureGgapExtract(RNAistructure, StructureGgaptotal)
#     seq = SequenceRNAikmer + SequenceRNAiGgap
#     strc = StructureRNAikmer + StructureRNAiGgap
#     Seq_j.append(f"{seq}")
#     Str_j.append(f"{strc}")
#     # 3_mer + Word2Vec：32维
#     k_word2vec = kmer_word2vec(RNAisequence, 3)
#     w2v_j.append(f"{k_word2vec}")
    # # GC特征：7维    初始化GCconder类，提供序列信息
    # gc_calculator = GCconder(sequence=RNAisequence)
    # gc = gc_calculator.get_gc()
    # gc = ' '.join(map(str, gc))
    # GC_j.append(gc)
    # # 转录序列描述CTD：30维
    # ctd_calculator = CTDcoder(sequence=RNAisequence)
    # ctd = ctd_calculator.CTD()
    # ctd = ' '.join(map(str, ctd))
    # CTD_j.append(f"{ctd}")
    # # 伪蛋白特征： 5维 (需要后续进行归一化处理，保持量纲统一)
    # protpar = ProtPar(RNAisequence)
    # pro_fea = protpar.get_features()
    # Protpar_j.append(pro_fea)

# scaler = MinMaxScaler()
# Protpar_j= scaler.fit_transform(Protpar_j)
# pro_str_j = [' '.join(map(str, features)) for features in Protpar_j]

# with open ('../data/miRNA/Seq.csv', 'w') as file:
#     for fea in Seq_j:
#         file.write(fea+'\n')
# with open ('../data/miRNA/Str.csv', 'w') as file:
#     for fea in Str_j:
#         file.write(fea+'\n')

# with open ('data/miRNA/Word2Vec_TF.csv','w') as file:
#     for fea in w2v_j:
#         file.write(fea+'\n')
# with open ('data/miRNA/GC_related.csv','w') as file1:
#     for gc in GC_j:
#         file1.write(gc+'\n')
# with open ('data/miRNA/CTD_sequence.csv','w') as file2:
#     for gc in CTD_j:
#         file2.write(gc+'\n')
# with open ('data/miRNA/Protpar.csv','w') as file3:
#     for pro in pro_str_j:
#         file3.write(pro+'\n')
# with open ('data/miRNA/Feature_j.csv','w') as file4:
#     for pro in Feature_j:
#         file4.write(pro+'\n')


RP11-654A16.3
LINC00963
LINC00588
LINC00599
RP11-217B7.2
SMG7-AS1
ACVR2B-AS1
ATXN8OS
GS1-600G8.3
AC007405.4
LINC00574
LINC00472
RP11-490O6.2
MCF2L-AS1
RP11-473M20.14
LINC00939
RP11-65G9.1
ARAP2
RP1-28O17.1
CASP8AP2
RCL1
FTX
IL12A-AS1
AC004237.1
RP11-314A20.2
RGPD4-AS1
RP11-84C10.2
PEX5L-AS2
LINC00266-1
RP4-792G4.2
CTA-211A9.5
RP3-525N10.2
LINC00309
LINC00483
RP11-25K19.1
TAS2R1
AC013275.2
SHANK2-AS3
RP11-263K19.4
RP11-359K18.3
RP11-544L8__B.4
TTTY14
DDX11-AS1
RP11-463J7.2
CTD-2201E18.3
RP11-998D10.4
AP001604.3
RP11-143E21.6
GS1-24F4.2
AP000345.1
LINC00314
LINC00305
AC025335.1
LINC00652
TTTY11
RP4-539M6.14
FAM225A
RP11-834C11.4
ATG9B
LINC00471
DLEU2
LINC00301
RP11-10L12.4
HOXB7
LINC00482
RP11-231E19.1
RP1-46F2.2
LINC00210
MATN1-AS1
MIR22HG
NEBL-AS1
RP11-161I2.1
RP11-53B5.1
LINC00319
RP11-431M7.3
RP11-756H20.1
AC069277.2
AC005772.2
LINC00615
LINC00867
LINC00708
RP11-425I13.3
GATA3-AS1
RP11-421F16.3
FLVCR1-AS1
RP3-337H4.8
Z99756.1
RP11-351J23.1
CTD-2313J17.5
AC079586.1
RP11-638F5.1
RP5-96

In [5]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

pathi='二级结构.csv'   # lncRNA存放格式: 名称, 人类的序列, 二级结构
list_tv_set = open(pathi, 'r').readlines()[1:]  # 跳过表头

CTD = []
Protpar = []
GC = []
w2v = []
Feature = []
lncRNA_names = []
sequences = []
structures = []

for line in list_tv_set:
    RNAiname, RNAisequence, RNAistructure = line.strip().split(',')
    lncRNA_names.append(RNAiname)
    sequences.append(RNAisequence)
    structures.append(RNAistructure)

    # RNAi sequence k-mer
    SequenceRNAikmer = SequencekmerExtract(RNAisequence, Sequencekmertotal)
    # RNAi sequence g-gap
    SequenceRNAiGgap = SequenceGgapExtract(RNAisequence, SequenceGgaptotal)
    # RNAi structure k-mer
    StructureRNAikmer = StructurekmerExtract(RNAistructure, Structurekmertotal)
    # RNAi structure g-gap
    StructureRNAiGgap = StructureGgapExtract(RNAistructure, StructureGgaptotal)

    featurei = SequenceRNAikmer + SequenceRNAiGgap + StructureRNAikmer + StructureRNAiGgap
    Feature.append(f"{featurei}")

    # 3_mer + Word2Vec：32维
    k_word2vec = kmer_word2vec(RNAisequence, 3)
    w2v.append(f"{k_word2vec}")

    # GC特征
    gc_calculator = GCconder(sequence=RNAisequence)
    gc = gc_calculator.get_gc()
    GC.append(gc)

    # CTD特征
    ctd_calculator = CTDcoder(sequence=RNAisequence)
    ctd = ctd_calculator.CTD()
    CTD.append(ctd)

    # 伪蛋白特征
    protpar = ProtPar(RNAisequence)
    pro_fea = protpar.get_features()
    Protpar.append(pro_fea)

    print(RNAiname)

# 归一化 Protpar
scaler = MinMaxScaler()
Protpar = scaler.fit_transform(Protpar)

# ---------------------------------------------
# 原有特征保存逻辑
# ---------------------------------------------
with open('Word2Vec.csv','a') as file:
    for fea in w2v:
        file.write(fea+'\n')

with open('GC_related.csv','a') as file1:
    for gc in GC:
        file1.write(' '.join(map(str, gc)) + '\n')

with open('CTD_sequence.csv','a') as file2:
    for ctd in CTD:
        file2.write(' '.join(map(str, ctd)) + '\n')

with open('Protpar.csv','a') as file3:
    for pro in Protpar:
        file3.write(' '.join(map(str, pro)) + '\n')

with open('Feature_i.csv','a') as file4:
    for pro in Feature:
        file4.write(pro+'\n')

# ---------------------------------------------
# 新增：把所有特征整合到原二级结构文件中，并保存为一个新的 CSV
# ---------------------------------------------
# Word2Vec
w2v_list = [list(map(float, fea.split())) for fea in w2v]
w2v_df = pd.DataFrame(w2v_list, columns=[f"w2v_{i}" for i in range(len(w2v_list[0]))])

# GC
gc_df = pd.DataFrame(GC, columns=[f"gc_{i}" for i in range(len(GC[0]))])

# CTD
ctd_df = pd.DataFrame(CTD, columns=[f"ctd_{i}" for i in range(len(CTD[0]))])

# Protpar
protpar_df = pd.DataFrame(Protpar, columns=[f"protpar_{i}" for i in range(Protpar.shape[1])])

# Feature
feature_list = [list(map(float, fea.split())) for fea in Feature]
feature_df = pd.DataFrame(feature_list, columns=[f"feature_{i}" for i in range(len(feature_list[0]))])

# 原始二级结构数据
df = pd.DataFrame({
    "lncRNA": lncRNA_names,
    "Sequence": sequences,
    "Structure": structures
})

# 合并所有特征
final_df = pd.concat([df, w2v_df, gc_df, ctd_df, protpar_df, feature_df], axis=1)

# 保存为新的 CSV
final_df.to_csv("lncRNA_features_all.csv", index=False)


RP11-654A16.3
LINC00963
LINC00588
LINC00599
RP11-217B7.2
SMG7-AS1
ACVR2B-AS1
ATXN8OS
GS1-600G8.3
AC007405.4
LINC00574
LINC00472
RP11-490O6.2
MCF2L-AS1
RP11-473M20.14
LINC00939
RP11-65G9.1
ARAP2
RP1-28O17.1
CASP8AP2
RCL1
FTX
IL12A-AS1
AC004237.1
RP11-314A20.2
RGPD4-AS1
RP11-84C10.2
PEX5L-AS2
LINC00266-1
RP4-792G4.2
CTA-211A9.5
RP3-525N10.2
LINC00309
LINC00483
RP11-25K19.1
TAS2R1
AC013275.2
SHANK2-AS3
RP11-263K19.4
RP11-359K18.3
RP11-544L8__B.4
TTTY14
DDX11-AS1
RP11-463J7.2
CTD-2201E18.3
RP11-998D10.4
AP001604.3
RP11-143E21.6
GS1-24F4.2
AP000345.1
LINC00314
LINC00305
AC025335.1
LINC00652
TTTY11
RP4-539M6.14
FAM225A
RP11-834C11.4
ATG9B
LINC00471
DLEU2
LINC00301
RP11-10L12.4
HOXB7
LINC00482
RP11-231E19.1
RP1-46F2.2
LINC00210
MATN1-AS1
MIR22HG
NEBL-AS1
RP11-161I2.1
RP11-53B5.1
LINC00319
RP11-431M7.3
RP11-756H20.1
AC069277.2
AC005772.2
LINC00615
LINC00867
LINC00708
RP11-425I13.3
GATA3-AS1
RP11-421F16.3
FLVCR1-AS1
RP3-337H4.8
Z99756.1
RP11-351J23.1
CTD-2313J17.5
AC079586.1
RP11-638F5.1
RP5-96

In [7]:
import pandas as pd
from gensim.models import Word2Vec
import numpy as np

# ---------- 设置参数 ----------
pathi = '二级结构.csv'  # 输入文件，格式：lncRNA,Sequence,Structure
output_file = 'Word2Vec2.csv'  # 输出文件
k = 3  # k-mer长度
vector_size = 128  # Word2Vec向量维度
window = 5
epochs = 50

# ---------- 函数定义 ----------
def get_kmers(sequence, k):
    """生成序列的k-mer列表"""
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

def get_sequence_vector(sequence, model, k):
    """根据Word2Vec模型将序列转换为向量"""
    kmers = get_kmers(sequence, k)
    vector_size = model.vector_size
    seq_vector = np.zeros(vector_size)
    count = 0
    for kmer in kmers:
        if kmer in model.wv:
            seq_vector += model.wv[kmer]
            count += 1
    if count > 0:
        seq_vector /= count
    return seq_vector

def kmer_word2vec(seq, k, vector_size=128, window=5, epochs=50):
    """生成序列Word2Vec特征"""
    # 保证长度能被3整除（原脚本逻辑）
    seq = seq[:-(len(seq) % 3)] if len(seq) % 3 != 0 else seq
    kmers = get_kmers(seq, k)
    sentences = [kmers]
    
    # 训练Word2Vec模型
    model = Word2Vec(vector_size=vector_size, window=window, min_count=1, sg=1, negative=5)
    model.build_vocab(sentences)
    model.train(sentences, total_examples=model.corpus_count, epochs=epochs)
    
    # 获取序列向量
    seq_vector = get_sequence_vector(seq, model, k)
    return ' '.join(map(str, seq_vector))

# ---------- 读取数据 ----------
df = pd.read_csv(pathi)
sequences = df['Sequence'].tolist()

# ---------- 生成Word2Vec特征 ----------
w2v_features = []
for seq in sequences:
    w2v_features.append(kmer_word2vec(seq, k))

# ---------- 保存到CSV ----------
with open(output_file, 'w', encoding='utf-8') as f:
    for feature in w2v_features:
        f.write(feature + '\n')

print(f"Word2Vec特征已保存到 {output_file}")


Word2Vec特征已保存到 Word2Vec2.csv


In [1]:
import pandas as pd

# 原始的 lncRNA 名称文件（注意 header=0）
name_file = "二级结构.csv"
df_name = pd.read_csv(name_file, header=0)  # 第一行是表头
lnc_names = df_name.iloc[:, 0]  # 第一列是 lncRNA 名称

# 五个特征文件
files = [
    "Word2Vec.csv",
    "GC_related.csv",
    "CTD_sequence.csv",
    "Protpar.csv",
    "Feature_i.csv"
]

# 读取所有特征文件
feature_lists = []
for path in files:
    df = pd.read_csv(path, header=None)
    feature_lists.append(df[0].astype(str))  # 转成字符串，方便拼接
    dim = len(df.iloc[0, 0].split())
    print(f"  {path}: {dim} 维")

# 按行拼接
merged_features = []
for i in range(len(lnc_names)):
    combined = " ".join([fea[i] for fea in feature_lists])
    merged_features.append(combined)

# 构建最终 DataFrame
final_df = pd.DataFrame({
    "lncRNA": lnc_names,
    "features": merged_features
})

# 保存
final_df.to_csv("All_Features_Concat.csv", index=False)
print("已保存到 All_Features_Concat.csv")


  Word2Vec.csv: 128 维
  GC_related.csv: 7 维
  CTD_sequence.csv: 30 维
  Protpar.csv: 5 维
  Feature_i.csv: 194 维
已保存到 All_Features_Concat.csv
